# Clean avalanche records

Takes the raw `records` table (produced by `01_ingestion/avalanche_reports.py`
via `transform.py`) and shapes it into an analysis-ready table: lower-cased
column names, a real `dato` dtype with derived `år`/`måned`/`dag` columns
(Norwegian names), the three count columns as `int64`, and the `source`
tagging column dropped.

Saved as `snow_avalanche_data` — a page can load it with
`storage.load("snow_avalanche_data")`.

In [1]:
import sys

sys.path.insert(0, "..")  # the notebook runs from 03_notebooks/

import pandas as pd

from backend import storage

storage.tables()

['records', 'snow_avalanche_data', 'trend_totals']

In [2]:
records = storage.load("records")
records.head()

,Dato,Døde,Kun skadet,Skredtatte,Sted,Latitude,Longitude,Kommune,Område,Aktivitet,Utløser,Bakkeaktivitet,Skredutstyr,Skredtype,Svakt lag,Skredstørrelse,Eksposisjon,Comment,source
0,2026-05-17,0,0.0,1,Rundfjellet,69.565128,19.299435,Tromsø,Troms,Topptur,Personutløst,Nedover,Ja,Løssnøskred,Ingen (løssnøskred),Ukjent,V,En person tatt av snøskred under nedkjøring. I...,avalanche_reports
1,2026-05-17,0,0.0,1,Tjønnholstinden,61.443767,8.650444,Vågå,Oppland,Ukjent,Personutløst,Ukjent,Ukjent,Flakskred,Ukjent,Størrelse 3,SØ,En person tatt av skred. Ikke meldt om skade.,avalanche_reports
2,2026-05-16,0,0.0,1,Storsteinnestinden,69.673372,18.499346,Tromsø,Troms,Topptur,Ukjent,Ukjent,Ukjent,Ukjent,Ukjent,Ukjent,V,En person ble tatt av snøskred. Vedkommende ko...,avalanche_reports
3,2026-05-14,0,0.0,2,Jægervasstindan,69.702153,20.027998,Lyngen,Troms,Topptur,Personutløst,Ukjent,Ja,Løssnøskred,Ukjent,Størrelse 1,V,Personutløst skred hvor to personer ble tatt. ...,avalanche_reports
4,2026-05-01,0,2.0,7,Konowfjellet,78.548386,12.974510,Svalbard,Svalbard,Topptur,Fjernutløst av person,Oppover,Nei,Flakskred,Ukjent,Størrelse 2,NV,Fjernutløst skred hvor syv personer ble tatt o...,avalanche_reports


Clean `records` into `snow_avalanche_data`:

1. Keep only the `avalanche_reports` rows (`records` also has `ice_reports`
   rows now — mixed in by `transform.py` stacking every raw source into one
   table), and drop any column that's entirely empty once filtered (the
   ice-only columns).
2. Lower-case every column name.
3. Parse `dato` to a real `datetime64` dtype (it comes in as ISO `YYYY-MM-DD`
   text), then derive `år` (year), `måned` (Norwegian month name) and `dag`
   (Norwegian weekday name) from it.
4. Cast `døde`, `kun skadet` and `skredtatte` to `int64` — a handful of
   records have no `kun skadet` figure at all; treated as zero rather than
   unknown, since that's what the site itself means by an absent count.
5. Drop rows where `område` is `Svalbard` — the map page's Kartverket tiles
   only cover mainland Norway, not Svalbard or Jan Mayen, so a Svalbard
   marker would sit on a blank/unrendered part of the map.
6. Drop the `source` column — it only mattered for tagging raw files before
   this cleaning step.

In [ ]:
NORWEGIAN_MONTHS = {
    1: "Januar", 2: "Februar", 3: "Mars", 4: "April", 5: "Mai", 6: "Juni",
    7: "Juli", 8: "August", 9: "September", 10: "Oktober", 11: "November", 12: "Desember",
}
NORWEGIAN_WEEKDAYS = {
    0: "Mandag", 1: "Tirsdag", 2: "Onsdag", 3: "Torsdag", 4: "Fredag", 5: "Lørdag", 6: "Søndag",
}

clean = pd.DataFrame()

if records.empty or "source" not in records.columns or "avalanche_reports" not in records["source"].values:
    print("No `avalanche_reports` rows in `records` yet — run `make pipeline` first.")
else:
    clean = records[records["source"] == "avalanche_reports"].dropna(axis=1, how="all")
    clean.columns = clean.columns.str.lower()

    clean["dato"] = pd.to_datetime(clean["dato"], format="%Y-%m-%d")
    clean["år"] = clean["dato"].dt.year
    clean["måned"] = clean["dato"].dt.month.map(NORWEGIAN_MONTHS)
    clean["dag"] = clean["dato"].dt.weekday.map(NORWEGIAN_WEEKDAYS)

    count_columns = ["døde", "kun skadet", "skredtatte"]
    clean[count_columns] = clean[count_columns].fillna(0).astype("int64")

    # Svalbard has real accidents in the data, but the map page's Kartverket
    # tiles only cover mainland Norway (verified: Svalbard/Jan Mayen tiles
    # come back blank) — drop so every remaining marker lands on real terrain.
    clean = clean[clean["område"] != "Svalbard"]

    clean = clean.drop(columns="source")

    storage.save("snow_avalanche_data", clean)

clean.head()